# Phase 4 — Feature Engineering & Dimensionality Reduction

**Project:** Astronomical Object Classification Using Machine Learning  
**Dataset:** SDSS Stellar Classification Dataset (SDSS17) — Cleaned Data  
**Domain:** Astronomy and Space Science  

### Context & Scientific Motivation
Raw photometric measurements in astronomy ($u, g, r, i, z$) represent apparent brightness across ultraviolet, optical, and near-infrared filter bands. However, apparent brightness is heavily confounded by distance: an intrinsically luminous galaxy at high redshift can have the same apparent magnitude as an intrinsically faint star nearby in the Milky Way. 

In astronomical science, the fundamental physical signature that separates object classes is the **Spectral Energy Distribution (SED)**, which is captured photometrically through **Colour Indices** (the logarithmic difference in flux between two passbands). By computing both adjacent and wide colour indices, second-order spectral curvatures, and aggregate magnitude dispersion, we extract distance-invariant physical features. Furthermore, aligned with the **DS605 Course Syllabus** (*Dimensionality Reduction using PCA and SVD*), we apply Principal Component Analysis (PCA) to evaluate low-dimensional embeddings and assess class separability.

## 1. Imports, Paths, and Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.model_selection import train_test_split

# Aesthetic plot styling for academic figures
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

# Resolve project root robustly
current_dir = Path.cwd()
ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
DATA_PATH = ROOT / 'data' / 'processed' / 'sdss_cleaned.csv'
FIGURES_DIR = ROOT / 'figures' / 'feature_engineering'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root resolved to: {ROOT}")
print(f"Reading cleaned data from: {DATA_PATH}")

Project root resolved to: c:\Users\Akanksha\OneDrive\Desktop\ML_PROJECT2\ML Project\astronomy-ml-classification
Reading cleaned data from: c:\Users\Akanksha\OneDrive\Desktop\ML_PROJECT2\ML Project\astronomy-ml-classification\data\processed\sdss_cleaned.csv


## 2. Load Phase 1 Cleaned Data
We load the verified clean dataset (99,999 observations, 0 missing values, -9999 sentinel removed).

In [2]:
df = pd.read_csv(DATA_PATH, dtype={'obj_ID': 'string', 'spec_obj_ID': 'string'})
print(f"Loaded shape: {df.shape}")
print(f"Target class counts:\n{df['class'].value_counts()}\n")
df.head(3)

Loaded shape: (99999, 18)
Target class counts:
class
GALAXY    59445
STAR      21593
QSO       18961
Name: count, dtype: int64



## 3. Photometric Feature Engineering

We engineer four distinct families of features:
1. **Adjacent Colour Indices ($u-g, g-r, r-i, i-z$):** Standard astronomical color slopes between consecutive wavelength bands.
2. **Wide / Non-Adjacent Colour Indices ($u-r, u-z, g-z$):** Broad wavelength baselines. $u-r$ is especially important for galaxy bimodality (separating star-forming blue galaxies from quiescent red ellipticals).
3. **Spectral Curvatures ($u-2g+r$ and $g-2r+i$):** Second-order differences measuring spectral inflection (Balmer break and Paschen continuum).
4. **Photometric Dispersion Indicators:** Mean brightness across all bands and standard deviation ($\sigma_{mag}$), which captures overall spectral contrast.

In [3]:
# 1. Adjacent color indices
df['u_g'] = df['u'] - df['g']
df['g_r'] = df['g'] - df['r']
df['r_i'] = df['r'] - df['i']
df['i_z'] = df['i'] - df['z']

# 2. Wide spectral indices
df['u_r'] = df['u'] - df['r']
df['u_z'] = df['u'] - df['z']
df['g_z'] = df['g'] - df['z']

# 3. Second-order spectral curvatures
df['curvature_ugr'] = df['u_g'] - df['g_r']
df['curvature_gri'] = df['g_r'] - df['r_i']

# 4. Photometric dispersion statistics
raw_bands = ['u', 'g', 'r', 'i', 'z']
df['mag_mean'] = df[raw_bands].mean(axis=1)
df['mag_std'] = df[raw_bands].std(axis=1)

feature_cols = [
    'u', 'g', 'r', 'i', 'z',
    'u_g', 'g_r', 'r_i', 'i_z',
    'u_r', 'u_z', 'g_z',
    'curvature_ugr', 'curvature_gri',
    'mag_mean', 'mag_std'
]

print(f"Total engineered photometric features: {len(feature_cols)}")
df[['class'] + feature_cols[:6] + feature_cols[6:10]].head(3)

Total engineered photometric features: 16


## 4. Multicollinearity & Correlation Analysis
We compute the Pearson correlation matrix to understand inter-feature relationships and collinearity among adjacent filters and derived colors.

In [4]:
plt.figure(figsize=(12, 10))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, annot=True, fmt='.2f',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Photometric Feature Correlation Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_feature_correlation_matrix.png', dpi=300)
plt.show()

## 5. Principal Component Analysis (PCA) — Unsupervised Dimensionality Reduction

As specified in the DS605 Course Syllabus (*Unit: Dimensionality Reduction using PCA and SVD*), PCA projects high-dimensional data along orthogonal axes of maximum variance.
- Let $X \in \mathbb{R}^{n \times p}$ be the standardized data matrix with zero mean and unit variance.
- The sample covariance matrix is $\Sigma = \frac{1}{n-1} X^T X$.
- The principal directions are the eigenvectors $v_j$ satisfying $\Sigma v_j = \lambda_j v_j$, where $\lambda_j$ is the variance explained by the $j$-th principal component.

In [5]:
# Standardize features before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

pca = PCA(n_components=5, random_state=42)
X_pca = pca.fit_transform(X_scaled)

for i in range(3):
    df[f'pca_{i+1}'] = X_pca[:, i]

exp_var = pca.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

for i, (ev, cv) in enumerate(zip(exp_var, cum_var), start=1):
    print(f"PC{i}: Individual Variance = {ev*100:5.2f}%, Cumulative = {cv*100:5.2f}%")

PC1: Individual Variance = 41.18%, Cumulative = 41.18%
PC2: Individual Variance = 32.32%, Cumulative = 73.50%
PC3: Individual Variance = 13.61%, Cumulative = 87.11%
PC4: Individual Variance =  7.74%, Cumulative = 94.84%
PC5: Individual Variance =  4.94%, Cumulative = 99.78%


### Scree Plot & Cumulative Explained Variance

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
components = np.arange(1, len(exp_var) + 1)

ax1.bar(components, exp_var * 100, color='#2b5c8f', alpha=0.85, edgecolor='black')
ax1.plot(components, exp_var * 100, 'o-', color='#d97706', lw=2)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)')
ax1.set_title('PCA Scree Plot (Individual Variance)')
ax1.set_xticks(components)

ax2.plot(components, cum_var * 100, 's-', color='#059669', lw=2, markersize=8)
ax2.axhline(80, color='gray', linestyle='--', alpha=0.7, label='80% Variance Threshold')
ax2.axhline(90, color='red', linestyle='--', alpha=0.7, label='90% Variance Threshold')
ax2.set_xlabel('Number of Principal Components')
ax2.set_ylabel('Cumulative Variance Explained (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.set_xticks(components)
ax2.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_pca_scree_plot.png', dpi=300)
plt.show()

### 2D Projection in PCA Space (PC1 vs PC2)
We project celestial objects onto the first two principal component axes to evaluate visual separability.

In [7]:
palette = {'GALAXY': '#2b5c8f', 'STAR': '#d97706', 'QSO': '#7c3aed'}

plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=df.sample(6000, random_state=42),
    x='pca_1', y='pca_2', hue='class',
    palette=palette, alpha=0.45, s=20, edgecolor='none'
)
plt.title('SDSS Astronomical Objects in 2D PCA Space', fontsize=13)
plt.xlabel(f'Principal Component 1 ({exp_var[0]*100:.1f}% Variance)')
plt.ylabel(f'Principal Component 2 ({exp_var[1]*100:.1f}% Variance)')
plt.xlim(-8, 8)
plt.ylim(-6, 6)
plt.legend(title='Celestial Class', frameon=True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_pca_2d_projection.png', dpi=300)
plt.show()

## 6. Feature Importance: ANOVA F-Score & Mutual Information

We perform statistical ranking of engineered features:
1. **ANOVA F-statistic (Linear separability):** Ratio of between-group variance to within-group variance ($F = \frac{MS_{between}}{MS_{within}}$).
2. **Mutual Information (Non-linear association):** Information-theoretic measure of dependency $I(X; Y) = \iint p(x,y) \log \frac{p(x,y)}{p(x)p(y)} dx dy$, capturing non-linear relationships without parametric assumptions.

In [8]:
# Stratified sample of 15,000 for efficient mutual information computation
sample_df, _ = train_test_split(df, train_size=15000, stratify=df['class'], random_state=42)
X_sample = sample_df[feature_cols]
y_sample = sample_df['class']

f_scores, f_pvalues = f_classif(X_sample, y_sample)
mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42)

feature_ranking = pd.DataFrame({
    'Feature': feature_cols,
    'ANOVA_F_Score': f_scores,
    'ANOVA_p_value': f_pvalues,
    'Mutual_Information': mi_scores
}).sort_values(by='Mutual_Information', ascending=False).reset_index(drop=True)

print("Top 10 Features Ranked by Mutual Information:")
feature_ranking.head(10)

Top 10 Features Ranked by Mutual Information:


### Feature Importance Barplot

In [9]:
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_ranking, x='Mutual_Information', y='Feature', hue='Feature', palette='Blues_r', legend=False)
plt.title('Photometric Feature Ranking via Mutual Information', fontsize=13)
plt.xlabel('Mutual Information with Astronomical Class')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_mutual_information_ranking.png', dpi=300)
plt.show()

## 7. Export Processed Feature Dataset
We export the completed dataset with engineered features to `data/processed/sdss_features.csv` for Phase 5 model training.

In [10]:
export_cols = ['obj_ID', 'alpha', 'delta'] + feature_cols + ['pca_1', 'pca_2', 'pca_3', 'redshift', 'class']
output_path = ROOT / 'data' / 'processed' / 'sdss_features.csv'
df[export_cols].to_csv(output_path, index=False)
print(f"Successfully saved {len(df)} rows with {len(export_cols)} columns to {output_path}.")

Successfully saved 99999 rows with 24 columns to c:\Users\Akanksha\OneDrive\Desktop\ML_PROJECT2\ML Project\astronomy-ml-classification\data\processed\sdss_features.csv.


## 8. Summary of Phase 4 Findings & Scientific Interpretation

1. **Dominance of Colour Indices:** As predicted by astrophysical theory, the engineered colour indices (`r_i`, `g_z`, `g_r`, `u_z`, `u_r`) rank substantially higher in mutual information than any raw single-band apparent magnitudes ($u, g, r, i, z$). This confirms that the spectral slope (SED) is far more discriminative of celestial object type than total apparent brightness.
2. **Dimensionality Reduction:** The first 3 principal components capture **87.1%** of the total variance across all 16 photometric attributes, and 5 components capture **99.8%**.
3. **Physical Interpretation:** 
   - `r-i` and `g-r` effectively separate cool stars and red early-type galaxies from blue objects.
   - `u-g` captures the classic UV-excess of Quasars ($z \lesssim 2.2$).
   - `mag_std` serves as an effective proxy for spectral curvature and variance across filters.
4. **Handoff to Phase 5:** We now proceed to train and evaluate supervised classification models (Logistic Regression, kNN, Support Vector Machines, Decision Trees, and Ensembles) using leak-safe cross-validation pipelines.